In [72]:
# Importing Libraries
import numpy as np
import pandas as pd
import cloudscraper
import random
import time
from io import StringIO

scraper = cloudscraper.create_scraper()

In [73]:
teams = [
    'ATL', 'BOS', 'BRK', 'CHO', 'CHI', 'CLE', 'DAL', 'DEN', 'DET', 'GSW',
    'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 'NOP', 'NYK',
    'OKC', 'ORL', 'PHI', 'PHO', 'POR', 'SAC', 'SAS', 'TOR', 'UTA', 'WAS'
]
len(teams)

30

In [74]:
team_dict = {
    'ATL': 'Hawks',
    'BOS': 'Celtics',
    'BRK': 'Nets',
    'CHO': 'Hornets',
    'CHI': 'Bulls',
    'CLE': 'Cavaliers',
    'DAL': 'Mavericks',
    'DEN': 'Nuggets',
    'DET': 'Pistons',
    'GSW': 'Warriors',
    'HOU': 'Rockets',
    'IND': 'Pacers',
    'LAC': 'Clippers',
    'LAL': 'Lakers',
    'MEM': 'Grizzlies',
    'MIA': 'Heat',
    'MIL': 'Bucks',
    'MIN': 'Timberwolves',
    'NOP': 'Pelicans',
    'NYK': 'Knicks',
    'OKC': 'Thunder',
    'ORL': 'Magic',
    'PHI': '76ers',
    'PHO': 'Suns',
    'POR': 'Trail Blazers',
    'SAC': 'Kings',
    'SAS': 'Spurs',
    'TOR': 'Raptors',
    'UTA': 'Jazz',
    'WAS': 'Wizards',
}
len(team_dict)

30

In [75]:
seasons = ['2026']
len(seasons)

1

In [76]:
# create empty dataframe
nba_df = pd.DataFrame()

# iterate through seasons
for season in seasons:

    # iterate through teams
    for team in teams:

        url = 'https://www.basketball-reference.com/teams/' + team + "/" + season + '/gamelog-advanced/'
        print(url)

        # get response object
        response = scraper.get(url)

        # remove blocker tag for playoff table
        visible_html = response.text.replace('<!--', '')

        # get dataframes from tables using visible HTML text, keep only game log tables
        all_tables = pd.read_html(StringIO(visible_html), header=1)
        df_list = [t for t in all_tables if 'Gtm' in t.columns]

        # skip if no valid tables found
        if len(df_list) == 0:
            print(f"No game log table found for {team} {season}, skipping.")
            continue
        # if no playoffs
        elif len(df_list) == 1:
            team_df = df_list[0]
        # if playoffs, combine reg season and playoff table
        else:
            df_list[1]['Gtm'] = df_list[1]['Gtm'] + 82
            team_df = pd.concat([df_list[0], df_list[1]], ignore_index=True)

        # drop rows where 'Gtm' is NaN or equal to 'Gtm'
        team_df = team_df.dropna(subset=['Gtm'])
        team_df = team_df[team_df['Gtm'] != 'Gtm']
        team_df = team_df.reset_index(drop=True)

        # keep columns 1 - 11
        team_df = team_df.iloc[:, 1:11]

        # rename some columns
        team_df = team_df.rename(columns={'Unnamed: 3': 'At', 'Tm': 'OPts', 'Opp.1': 'Dpts'})

        # replace values in columns 'At' and 'Rslt'
        team_df['At'] = team_df['At'].apply(lambda x: x if x == '@' else "")
        team_df['Win'] = team_df['Rslt'].apply(lambda x: 1 if str(x).startswith('W') else 0)
        team_df = team_df.drop(columns=['Rslt'])

        # add columns to the team_df
        team_df.insert(loc=0, column='Season', value=season)
        team_df.insert(loc=2, column='Team', value=team)
        team_df.insert(loc=3, column='Name', value=team_dict[team])
        team_df['Location'] = team_df['At'].apply(lambda x: 'Away' if x == '@' else 'Home')
        team_df.insert(loc=6, column='Location', value=team_df.pop('Location'))
        team_df['Opp_Name'] = team_df['Opp'].apply(lambda x: team_dict[x])
        team_df.insert(loc=8, column='Opp_Name', value=team_df.pop('Opp_Name'))

        # append current year and team gamelogs to the aggregate dataframe
        nba_df = pd.concat([nba_df, team_df], ignore_index=True)

        # pause program to go by basketball-reference.com rules
        time.sleep(random.randint(4, 6))

# display aggregate dataframe
print(nba_df.shape)

https://www.basketball-reference.com/teams/ATL/2026/gamelog-advanced/
https://www.basketball-reference.com/teams/BOS/2026/gamelog-advanced/
https://www.basketball-reference.com/teams/BRK/2026/gamelog-advanced/
https://www.basketball-reference.com/teams/CHO/2026/gamelog-advanced/
https://www.basketball-reference.com/teams/CHI/2026/gamelog-advanced/
https://www.basketball-reference.com/teams/CLE/2026/gamelog-advanced/
https://www.basketball-reference.com/teams/DAL/2026/gamelog-advanced/
https://www.basketball-reference.com/teams/DEN/2026/gamelog-advanced/
https://www.basketball-reference.com/teams/DET/2026/gamelog-advanced/
https://www.basketball-reference.com/teams/GSW/2026/gamelog-advanced/
https://www.basketball-reference.com/teams/HOU/2026/gamelog-advanced/
https://www.basketball-reference.com/teams/IND/2026/gamelog-advanced/
https://www.basketball-reference.com/teams/LAC/2026/gamelog-advanced/
https://www.basketball-reference.com/teams/LAL/2026/gamelog-advanced/
https://www.basketba